In [2]:
import numpy as np
import gudhi as gd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from scipy.sparse import dok_matrix
import pandas as pd

def load_distance_matrix(file_name):
    return pd.read_excel(file_name, index_col=0).values

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    n = distance_matrix.shape[0]
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
    return simplex_tree

# Function to plot the persistence diagram
def plot_persistence_diagram(simplex_tree):
    persistence = simplex_tree.persistence()
    gd.plot_persistence_diagram(persistence)
    plt.title("Persistence Diagram")
    plt.show()

# Extract simplices from filtration
def extract_simplices(filtration):
    simplices = {'vertices': [], 'edges': [], 'faces': []}
    for simplex, _ in filtration:
        if len(simplex) == 1:
            simplices['vertices'].append(simplex)
        elif len(simplex) == 2:
            simplices['edges'].append(simplex)
        elif len(simplex) == 3:
            simplices['faces'].append(simplex)
    return simplices

# Function to get the boundary matrix
def get_boundary_matrix(simplices_higher, simplices_lower):
    if not simplices_higher or not simplices_lower:
        return None
    n_higher = len(simplices_higher)
    n_lower = len(simplices_lower)
    boundary_matrix = dok_matrix((n_lower, n_higher), dtype=int)

    simplex_to_index = {tuple(simplex): i for i, simplex in enumerate(simplices_lower)}

    for j, higher_simplex in enumerate(simplices_higher):
        for i, vertex in enumerate(higher_simplex):
            boundary_simplex = tuple(sorted(higher_simplex[:i] + higher_simplex[i+1:]))
            if boundary_simplex in simplex_to_index:
                row_index = simplex_to_index[boundary_simplex]
                boundary_matrix[row_index, j] = (-1) ** i

    return boundary_matrix

# Compute Laplacian
def compute_laplacian(boundary_prev, boundary_next):
    laplacian = None
    if boundary_prev is not None:
        laplacian = boundary_prev.transpose().dot(boundary_prev)
    if boundary_next is not None:
        if laplacian is None:
            laplacian = boundary_next.dot(boundary_next.transpose())
        else:
            laplacian += boundary_next.dot(boundary_next.transpose())
    return laplacian

# Compute eigenvalues
def compute_eigenvalues(laplacian):
    if laplacian is not None:
        _, singular_values, _ = np.linalg.svd(laplacian.toarray())
        return np.sort(singular_values)
    return None

# Function to visualize Rips complex
def visualize_rips_complex(radius):
    # Compute Rips complex
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, radius)
    filtration = list(simplex_tree.get_filtration())

    # Extract simplices
    simplices = extract_simplices(filtration)

    # Compute boundary matrices
    boundary_1 = get_boundary_matrix(simplices['edges'], simplices['vertices'])
    boundary_2 = get_boundary_matrix(simplices['faces'], simplices['edges'])

    # Compute Laplacians
    laplacian_0 = compute_laplacian(None, boundary_1)
    laplacian_1 = compute_laplacian(boundary_1, boundary_2) if boundary_1 is not None else None

    # Plot Persistence Diagram
    #plot_persistence_diagram(simplex_tree)

    # Print Laplacians and Eigenvalues
    if laplacian_0 is not None:
        print("\nLaplacian Matrix (L0):\n", laplacian_0.toarray())
        eigenvalues_0 = compute_eigenvalues(laplacian_0)
        print("Eigenvalues of L0:", eigenvalues_0)
        zero_eigenvalues_0 = np.sum(np.isclose(eigenvalues_0, 0))
        print("Number of zero eigenvalues of L0:", zero_eigenvalues_0)

    if laplacian_1 is not None:
        print("\nLaplacian Matrix (L1):\n", laplacian_1.toarray())
        eigenvalues_1 = compute_eigenvalues(laplacian_1)
        print("Eigenvalues of L1:", eigenvalues_1)
        zero_eigenvalues_1 = np.sum(np.isclose(eigenvalues_1, 0))
        print("Number of zero eigenvalues of L1:", zero_eigenvalues_1)

distance_matrix = pd.read_excel('trial_distance_matrix.xlsx', index_col=0).values
n =  distance_matrix.shape[0]

# Interactive widget for max edge length
radius_slider = widgets.FloatSlider(value=0.2, min=0.1, max=np.max(distance_matrix), step=0.01, description='Max Edge Length:')
widgets.interactive(visualize_rips_complex, radius=radius_slider)


interactive(children=(FloatSlider(value=0.2, description='Max Edge Length:', max=1.708115479295449, min=0.1, s…

In [ ]:
import numpy as np
import gudhi as gd
import matplotlib.pyplot as plt
from scipy.sparse import dok_matrix
import networkx as nx
import ipywidgets as widgets

# Function to compute Rips complex from a distance matrix
def compute_rips_from_distance_matrix(distance_matrix, max_edge_length):
    rips_complex = gd.RipsComplex(distance_matrix=distance_matrix, max_edge_length=max_edge_length)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=2)
    return simplex_tree

# Function to extract simplices
def extract_simplices(simplex_tree):
    simplices = {'vertices': set(), 'edges': [], 'faces': []}

    for simplex, _ in simplex_tree.get_filtration():
        if len(simplex) == 1:
            simplices['vertices'].add(simplex[0])
        elif len(simplex) == 2:
            simplices['edges'].append(tuple(simplex))
        elif len(simplex) == 3:
            simplices['faces'].append(tuple(simplex))

    return simplices

# Function to visualize the Rips complex
def visualize_rips_complex(radius):
    simplex_tree = compute_rips_from_distance_matrix(distance_matrix, radius)
    simplices = extract_simplices(simplex_tree)

    # Create graph for visualization
    G = nx.Graph()
    G.add_nodes_from(simplices['vertices'])
    G.add_edges_from(simplices['edges'])

    # Compute layout using spring layout for better visualization
    pos = nx.spring_layout(G, seed=42)

    # Rename vertices as 'Protein X'
    labels = {i: f"Protein {i+1}" for i in simplices['vertices']}

    # Plot vertices
    plt.figure(figsize=(8, 6))
    nx.draw_networkx_nodes(G, pos, node_size=100, node_color="black")

    # Plot edges
    nx.draw_networkx_edges(G, pos, edgelist=simplices['edges'], width=2, edge_color="blue")

    # Plot filled triangles for 2-simplices
    for face in simplices['faces']:
        pts = np.array([pos[face[0]], pos[face[1]], pos[face[2]]])
        plt.fill(pts[:, 0], pts[:, 1], color='cyan', alpha=0.3)
    
    # Add labels
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_color="red")
    
    plt.title(f"Vietoris-Rips Complex (Max Edge = {radius:.2f})")
    plt.axis("off")
    plt.show()

# Interactive widget for max edge length
radius_slider = widgets.FloatSlider(value=0.2, min=0.1, max=np.max(distance_matrix), step=0.01, description='Max Edge Length:')
widgets.interactive(visualize_rips_complex, radius=radius_slider)

interactive(children=(FloatSlider(value=0.2, description='Max Edge Length:', max=1.708115479295449, min=0.1, s…